# AI GARP — Best Buy panel exploration

Starter notebook. Each cell is independent; run what you need.

Data path: `~/Dropbox/Apps/AIGarpDataPipeline/`. All loaders read from there by default.
See `DATA.md` at the project root for schema details.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from bbpipeline.analysis import (
    load_prices_latest, load_prices,
    load_metadata_current, load_metadata_at,
    load_panel, runs_summary, duckdb_connection,
)

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(10)

## 1 — Pipeline health

Per-run status. If this is empty or full of errors, the rest of the notebook won't make sense.

In [ ]:
runs_summary().select(
    ['started_at', 'status', 'products_fetched',
     'metadata_changed_rows', 'api_calls']
).sort('started_at', descending=True).head(10)

## 2 — Current catalog snapshot

Latest prices + some sanity checks.

In [ ]:
latest = load_prices_latest()
meta = load_metadata_current()

print(f'Active SKUs (prices):   {latest.height:,}')
print(f'Active SKUs (metadata): {meta.height:,}')
print(f'On sale right now:      '
      f'{latest.filter(pl.col("onSale")).height:,} '
      f'({100 * latest.filter(pl.col("onSale")).height / latest.height:.1f}%)')
print(f'Median regular price:   ${latest["regularPrice"].median():.2f}')

## 3 — Catalog by department

Which categories dominate.

In [ ]:
con = duckdb_connection()
con.execute('''
    SELECT m.department,
           COUNT(*) AS n_skus,
           AVG(p.regularPrice) AS mean_price,
           AVG(CASE WHEN p.onSale THEN 1.0 ELSE 0.0 END) AS pct_on_sale
    FROM prices p JOIN metadata_current m USING (sku)
    WHERE p.fetched_at = (SELECT MAX(fetched_at) FROM prices)
      AND m.department IS NOT NULL
    GROUP BY 1 ORDER BY n_skus DESC LIMIT 20
''').pl()

## 4 — Price distribution

Regular price (log scale; heavy right tail expected).

In [ ]:
prices = latest.filter(pl.col('regularPrice').is_not_null()
                        & (pl.col('regularPrice') > 0))['regularPrice'].to_numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(prices, bins=80, log=True, color='steelblue', edgecolor='white')
ax.set_xscale('log')
ax.set_xlabel('regular price ($, log scale)')
ax.set_ylabel('SKUs (log scale)')
ax.set_title('Best Buy catalog — regular price distribution')
plt.show()

## 5 — Price trajectory for a chosen SKU

Useful sanity check; also the atom for GARP-style analysis. Change `sku` to pick another product.

In [ ]:
# Pick a SKU — here, the most-reviewed product in the catalog
popular = (
    latest.filter(pl.col('customerReviewCount').is_not_null())
          .sort('customerReviewCount', descending=True)
          .head(1)
)
sku = popular['sku'][0]
name = meta.filter(pl.col('sku') == sku)['name'][0]
print(f'Trajectory for SKU {sku}: {name}')

traj = load_prices(skus=[sku]).sort('fetched_at')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(traj['fetched_at'], traj['regularPrice'], label='regular', marker='o')
ax.plot(traj['fetched_at'], traj['salePrice'],    label='sale',    marker='.')
ax.set_ylabel('price ($)')
ax.legend()
ax.set_title(name[:60])
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6 — GARP hook: relative prices for a choice set

For revealed-preference analysis over a fixed choice set, compute the relative price of each SKU against a baseline over time. Here we take the top 5 SKUs by review count, fix SKU 0 as baseline, and plot `p_i / p_0` over time.

This is a placeholder — substitute a research-relevant choice set once you have one.

In [ ]:
top5 = (
    latest.filter(pl.col('customerReviewCount').is_not_null())
          .sort('customerReviewCount', descending=True)
          .head(5)
)
top_skus = top5['sku'].to_list()

panel = load_prices(skus=top_skus).pivot(
    index='fetched_at', on='sku', values='salePrice'
).sort('fetched_at')

baseline = top_skus[0]
rel = panel.with_columns(
    [(pl.col(s) / pl.col(baseline)).alias(f'{s}/{baseline}') for s in top_skus[1:]]
)
rel.select(['fetched_at'] + [f'{s}/{baseline}' for s in top_skus[1:]]).head(20)

## 7 — Exports

Run from a terminal:

```bash
python scripts/export_csv.py metadata-current
python scripts/export_csv.py prices --start 2026-04-21 --end 2026-04-30
python scripts/export_csv.py panel  --start 2026-04-21
```

Defaults land in `./csv/`.